# Our heat-kernel operator vs SFNO on the sphere: same basis, different architecture

**Claim to test.** SFNO works in the spherical harmonic basis $Y_{lm}$. On the unit
sphere the spherical harmonics *are* the Laplace-Beltrami eigenfunctions,

$$-\Delta_{S^2} Y_{lm} = l(l+1)\,Y_{lm},$$

so if our architecture is given that exact eigenbasis it should be able to match SFNO.

**Setup, kept deliberately simple and fair:**

- **No mesh and no `robust_laplacian`.** Everything lives on a Gauss-Legendre
  grid (16 latitudes x 32 longitudes = 512 points). The basis is the 256 real
  spherical harmonics with $l \le 15$, eigenvalues $\lambda = l(l+1)$, and the
  quadrature weights act as the "mass". On this grid the harmonics are exactly
  orthonormal, $Y^\top W Y = I$.
- **Both models see the same grid, the same basis and the same data.**
  *Ours* is `IntegralOperatorModel` from `model.py`, unchanged; only its
  eigenpairs come from the harmonics instead of a mesh. *SFNO* is `SFNO` from
  `ref_models/sfno.py`, run directly on the grid with all degrees $l \le 15$, so
  there is no vertex/grid interpolation.
- **Why they are comparable.** SFNO's Driscoll-Healy spectral weight is one
  channel matrix $W_l$ per degree $l$. Our spectral multiplier is
  $\sum_r W_r e^{-t_r l(l+1)}$, which also depends only on $l$, so it is a
  smooth, low-parameter family of SFNO-style per-degree weights. By the addition
  theorem our heat kernel $K_t(x,y)=\sum_{lm} e^{-t l(l+1)} Y_{lm}(x)Y_{lm}(y)$
  is zonal: a function of the angle between $x$ and $y$ only. The remaining
  differences are architectural: our pairwise message $\phi(f(x), f(y))$ against
  SFNO's lifting, pointwise skips and decoder.

**Four problems** (input $u(0)$ field, target $u(T)$ field):

| | equation | type |
|---|---|---|
| ODE 1 | damped pendulum $\ddot x + c\dot x + \sin x = 0$ at every grid point | pointwise, nonlinear |
| ODE 2 | damped Duffing $\ddot x + c\dot x + x + x^3 = 0$ at every grid point | pointwise, nonlinear |
| PDE 1 | diffusion $u_t = D\,\Delta u$ | linear, nonlocal |
| PDE 2 | reaction-diffusion $u_t = D\,\Delta u + k u^2$ | nonlinear, nonlocal |

The equation classes come from `equations.py`. The ODEs start at rest with a
random smooth field as the initial position.

**Environment variables:** `N_EPOCHS` (default 1000), `OURS_MICRO_BATCH`
(default 16), `SMOKE=1` (tiny run to test the code).

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from equations import (
    Domain, Equation,
    DiffusionEquation, ReactionDiffusion,
    DrivenPendulum, DuffingOscillator,
)
from model import HeatKernel, IntegralOperator, IntegralOperatorModel
from ref_models.sfno import gauss_legendre_grid, real_spherical_harmonics, SFNO

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

## Configuration

In [ ]:
SEED = 0
N_TOTAL, N_TRAIN = 128, 96
N_EPOCHS = int(os.environ.get("N_EPOCHS", "1000"))
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

NLAT = 16                    # 16 x 32 Gauss-Legendre grid -> harmonics up to l = 15
LMAX = NLAT - 1

IC_SMOOTHING = 0.1           # IC spectrum ~ exp(-IC_SMOOTHING * l(l+1) / 2)

# Ours (same hyper-parameters as benchmark_manifolds_v2)
N_DIFFUSION_TIMES = 12
HIDDEN_CHANNELS = 32
LATENT_CHANNELS = 8
N_OPERATOR_LAYERS = 4
OURS_MICRO_BATCH = int(os.environ.get("OURS_MICRO_BATCH", "16"))

# SFNO: every degree the grid resolves gets a spectral weight
SFNO_MODES = LMAX
SFNO_WIDTH = 64

if os.environ.get("SMOKE") == "1":
    N_TOTAL, N_TRAIN, N_EPOCHS, BATCH_SIZE = 12, 8, 2, 4
    print("SMOKE TEST MODE")
print(f"N_EPOCHS = {N_EPOCHS}")

## The sphere: grid, spherical harmonics, Laplacian

`Y[:, k]` holds the harmonic $Y_{lm}$ sampled at the grid points, and `w` the
quadrature weights (summing to $4\pi$). The Laplacian on grid values is
$\Delta = -Y\,\mathrm{diag}(l(l+1))\,Y^\top W$, i.e. analyse, scale, synthesise.
It is exact for band-limited fields and replaces the mesh Laplacian. Wrapping it
in a `Domain` lets the PDE classes in `equations.py` run unchanged.

In [ ]:
theta, phi, w = gauss_legendre_grid(NLAT)                    # colatitude, longitude, weights
THETA, PHI = np.meshgrid(theta, phi, indexing="ij")
theta_grid, phi_grid = THETA.ravel(), PHI.ravel()

Y, degree = real_spherical_harmonics(LMAX, theta_grid, phi_grid)   # [V, (LMAX+1)^2]
eigenvalues = degree * (degree + 1.0)


class SphereGrid(Domain):
    def __init__(self):
        self.points = np.stack([np.sin(theta_grid) * np.cos(phi_grid),
                                np.sin(theta_grid) * np.sin(phi_grid),
                                np.cos(theta_grid)], axis=-1)
        self.n_points = len(theta_grid)
        self.laplacian = -(Y * eigenvalues) @ (Y * w[:, None]).T


sphere = SphereGrid()
V = sphere.n_points

print(f"grid {NLAT} x {2 * NLAT} = {V} points, {Y.shape[1]} harmonics (l <= {LMAX})")
print("orthonormality error  max|Y^T W Y - I| =",
      f"{np.abs(Y.T @ (w[:, None] * Y) - np.eye(Y.shape[1])).max():.1e}")
print("eigen-check           max|Lap Y_lm + l(l+1) Y_lm| =",
      f"{np.abs(sphere.laplacian @ Y + Y * eigenvalues).max():.1e}")

## Data

Initial conditions are random harmonic expansions with spectrum
$\propto e^{-s\,l(l+1)/2}$ (constant removed, max $|u|=1$). The ODEs are solved
independently at every grid point.

**Basis floor.** Our model's output is $Y\hat y$, so it lies in the span of the
256 harmonics. A pointwise nonlinear map such as $\sin$ or $x^3$ creates
harmonics above $l=15$, which our model cannot output, while SFNO's pointwise
layers can. The "outside basis" column below is that unavoidable relative error
for our model; `IC_SMOOTHING` and the amplitudes are chosen to keep it well
under 1% so it does not decide the comparison.

In [ ]:
class PointwiseODE(Equation):
    # An ODE from equations.py run independently at every grid point.
    # state = [x at all points, xdot at all points]
    def __init__(self, ode):
        super().__init__()
        self.ode = ode

    def evolution(self, t, state):
        return self.ode.evolution(t, state.reshape(2, -1)).ravel()


def random_field(rng):
    coefficients = rng.normal(size=len(degree)) * np.exp(-0.5 * IC_SMOOTHING * eigenvalues)
    coefficients[degree == 0] = 0.0
    field = Y @ coefficients
    return field / np.abs(field).max()


def solve_ode(ode, amplitude, T):
    def sample(rng):
        x0 = amplitude * random_field(rng)              # start at rest: xdot(0) = 0
        solution = PointwiseODE(ode).solve([0.0, T], state0=np.concatenate([x0, 0 * x0]),
                                           t_eval=[T], rtol=1e-8, atol=1e-8)
        return x0, solution.y[:V, -1]
    return sample


def solve_pde(equation, T):
    def sample(rng):
        u0 = random_field(rng)
        solution = equation.solve([0.0, T], state0=u0, t_eval=[T], rtol=1e-8, atol=1e-8)
        return u0, solution.y[:, -1]
    return sample


SYSTEMS = {
    "Pendulum (ODE)":           solve_ode(DrivenPendulum(c=0.2), amplitude=2.0, T=3.0),
    "Duffing (ODE)":            solve_ode(DuffingOscillator(c=0.2), amplitude=1.0, T=3.0),
    "Diffusion (PDE)":          solve_pde(DiffusionEquation(sphere, D=0.1), T=1.0),
    "Reaction-Diffusion (PDE)": solve_pde(ReactionDiffusion(sphere, D=0.1, k=0.5), T=1.0),
}


def weighted_norm(U):
    # quadrature (area-weighted) L2 norm over the grid, U: [..., V]
    return np.sqrt((w * U ** 2).sum())


def outside_basis(U):
    projection = (U @ (w[:, None] * Y)) @ Y.T
    return weighted_norm(U - projection) / weighted_norm(U)


raw_data = {}
t0 = time.time()
for name, sample in SYSTEMS.items():
    rng = np.random.default_rng(SEED)
    pairs = [sample(rng) for _ in range(N_TOTAL)]
    U0 = np.array([p[0] for p in pairs])
    UT = np.array([p[1] for p in pairs])
    raw_data[name] = (U0, UT)
    print(f"{name:26s} ||u(T)-u(0)||/||u(0)|| {weighted_norm(UT - U0) / weighted_norm(U0):.3f}   "
          f"u(T) outside basis (floor for Ours) {outside_basis(UT):.4f}   "
          f"finite {np.isfinite(UT).all()}")
print(f"data generated in {time.time() - t0:.1f}s")

### Normalisation and inputs

Training-set mean/std normalisation, as in the manifold benchmarks. Ours gets
the $xyz$ coordinates appended as channels; SFNO appends them itself inside
`SFNO.forward`, so both models see the same information.

In [ ]:
xyz = torch.tensor(sphere.points, dtype=torch.float32)

datasets = {}
for name, (U0, UT) in raw_data.items():
    X = torch.tensor(U0, dtype=torch.float32).unsqueeze(-1)       # [N, V, 1]
    Yt = torch.tensor(UT, dtype=torch.float32).unsqueeze(-1)
    X = (X - X[:N_TRAIN].mean()) / X[:N_TRAIN].std()
    Yt = (Yt - Yt[:N_TRAIN].mean()) / Yt[:N_TRAIN].std()
    datasets[name] = (X, Yt)


def model_input(model_name, X):
    if model_name == "Ours":
        return torch.cat([X, xyz.to(X.device).expand(X.shape[0], -1, -1)], dim=-1)
    return X

## Models

**Ours**: `HeatKernel` receives `eigenvectors = Y`, `eigenvalues = l(l+1)` and
`mass = w`, the exact sphere eigenpairs. Diffusion times are log-spaced from
$1/\lambda_{\max}$ to $3/\lambda_1$ as in `benchmark_manifolds_v2`.

**SFNO**: `ref_models/sfno.py` builds exactly the same `Y` and `w` internally
(same functions, same grid).

In [ ]:
def build_ours():
    times = np.logspace(np.log10(1.0 / eigenvalues.max()),
                        np.log10(3.0 / eigenvalues[eigenvalues > 0].min()),
                        N_DIFFUSION_TIMES)
    layers = []
    for i in range(N_OPERATOR_LAYERS):
        kernel = HeatKernel(
            eigenvalues=torch.tensor(eigenvalues, dtype=torch.float32),
            eigenvectors=torch.tensor(Y, dtype=torch.float32),
            mass=w,
            in_channels=4 if i == 0 else LATENT_CHANNELS,          # u + xyz
            out_channels=1 if i == N_OPERATOR_LAYERS - 1 else LATENT_CHANNELS,
            diffusion_times=times,
        )
        layers.append(IntegralOperator(kernel, hidden_channels=HIDDEN_CHANNELS))
    return IntegralOperatorModel(layers)


def build_sfno():
    return SFNO(nlat=NLAT, modes=SFNO_MODES, width=SFNO_WIDTH, in_channels=1, out_channels=1)


BUILDERS = {"Ours": build_ours, "SFNO": build_sfno}
MICRO_BATCH = {"Ours": OURS_MICRO_BATCH, "SFNO": BATCH_SIZE}

# sanity check: SFNO's internal basis is ours
assert np.allclose(build_sfno().conv0.synthesis.numpy(), Y.astype(np.float32))
for model_name, build in BUILDERS.items():
    print(f"{model_name:5s} parameters: {sum(p.numel() for p in build().parameters()):,}")

## Training

The training protocol is identical for both models: Adam at 1e-3 with the learning
rate halved every quarter of training. The loss is quadrature-weighted MSE (grid
points near the poles cover less area). The metric is the area-weighted relative
L2 error on the 32 held-out test samples.

In [ ]:
w_torch = torch.tensor(w, dtype=torch.float32)


def weighted_mse(prediction, target):
    return ((prediction - target) ** 2 * w_torch.to(target.device)[None, :, None]).sum() \
        / (target.shape[0] * w_torch.sum())


def relative_l2(prediction, target):
    weights = w_torch.to(target.device)[None, :, None]
    return torch.sqrt(((prediction - target) ** 2 * weights).sum()
                      / (target ** 2 * weights).sum()).item()


@torch.no_grad()
def predict(model, model_name, X):
    model.eval()
    return torch.cat([model(model_input(model_name, chunk.to(device))).cpu()
                      for chunk in X.split(MICRO_BATCH[model_name])])


def train(model_name, X, Yt):
    torch.manual_seed(SEED)
    model = BUILDERS[model_name]().to(device)
    X_train, Y_train = X[:N_TRAIN].to(device), Yt[:N_TRAIN].to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, N_EPOCHS // 4), gamma=0.5)
    history = {"train_loss": [], "test_error": []}

    for epoch in range(N_EPOCHS):
        model.train()
        permutation = torch.randperm(N_TRAIN, device=device)
        epoch_loss = 0.0
        for index in permutation.split(BATCH_SIZE):
            optimizer.zero_grad()
            for chunk in index.split(MICRO_BATCH[model_name]):
                loss = weighted_mse(model(model_input(model_name, X_train[chunk])), Y_train[chunk]) \
                    * len(chunk) / len(index)
                loss.backward()
                epoch_loss += loss.item()
            optimizer.step()
        scheduler.step()

        history["train_loss"].append(epoch_loss / len(permutation.split(BATCH_SIZE)))
        history["test_error"].append(relative_l2(predict(model, model_name, X[N_TRAIN:]), Yt[N_TRAIN:]))
    return model, history


results, histories, models = {}, {}, {}
for name, (X, Yt) in datasets.items():
    for model_name in BUILDERS:
        start = time.time()
        model, history = train(model_name, X, Yt)
        models[name, model_name], histories[name, model_name] = model, history
        results[name, model_name] = history["test_error"][-1]
        print(f"{name:26s} {model_name:5s} test rel L2 {history['test_error'][-1]:.4f}  "
              f"(best {min(history['test_error']):.4f}, {time.time() - start:.0f}s)", flush=True)

## Results

In [ ]:
print(f"{'system':26s}{'Ours':>10s}{'SFNO':>10s}{'Ours/SFNO':>11s}{'basis floor':>13s}")
for name in SYSTEMS:
    ours, sfno = results[name, "Ours"], results[name, "SFNO"]
    print(f"{name:26s}{ours:10.4f}{sfno:10.4f}{ours / sfno:11.2f}{outside_basis(raw_data[name][1]):13.4f}")

In [ ]:
colors = {"Ours": "#d62728", "SFNO": "#9467bd"}
fig, axes = plt.subplots(2, len(SYSTEMS), figsize=(4.2 * len(SYSTEMS), 7), sharex=True)
for column, name in enumerate(SYSTEMS):
    for model_name in BUILDERS:
        history = histories[name, model_name]
        axes[0, column].semilogy(history["train_loss"], color=colors[model_name], label=model_name)
        axes[1, column].semilogy(history["test_error"], color=colors[model_name], label=model_name)
    axes[0, column].set_title(name)
    axes[1, column].set_xlabel("epoch")
    for ax in axes[:, column]:
        ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("train weighted MSE")
axes[1, 0].set_ylabel("test relative L2")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

## Predictions on the sphere

One test sample per system: initial condition, true $u(T)$, both predictions, and
both pointwise errors, shown on a shared colour scale per row. The pole rows are
filled in from the nearest latitude ring, because Gauss-Legendre nodes do not
include the poles.

In [ ]:
def to_surface(values):
    # [V] grid values -> closed lat/lon surface (poles added, longitude wrapped)
    field = values.reshape(NLAT, 2 * NLAT)
    field = np.vstack([field[:1].mean() * np.ones((1, 2 * NLAT)), field,
                       field[-1:].mean() * np.ones((1, 2 * NLAT))])
    field = np.hstack([field, field[:, :1]])
    th = np.concatenate([[0.0], theta, [np.pi]])
    ph = np.concatenate([phi, [2 * np.pi]])
    TH, PH = np.meshgrid(th, ph, indexing="ij")
    return np.sin(TH) * np.cos(PH), np.sin(TH) * np.sin(PH), np.cos(TH), field


def sphere_plot(ax, values, vmin, vmax, cmap, title):
    x, y, z, field = to_surface(values)
    colors = plt.get_cmap(cmap)((field - vmin) / (vmax - vmin + 1e-12))
    ax.plot_surface(x, y, z, facecolors=colors, rstride=1, cstride=1, shade=False, linewidth=0)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=25, azim=-60)
    ax.set_axis_off()
    ax.set_title(title, fontsize=9)


sample = N_TRAIN          # first test sample
columns = ["u(0)", "true u(T)", "Ours", "SFNO", "|Ours - true|", "|SFNO - true|"]
fig = plt.figure(figsize=(2.9 * len(columns), 3.0 * len(SYSTEMS)))

for row, name in enumerate(SYSTEMS):
    X, Yt = datasets[name]
    truth = Yt[sample, :, 0].numpy()
    pred = {m: predict(models[name, m], m, X[sample:sample + 1])[0, :, 0].numpy() for m in BUILDERS}
    error = {m: np.abs(pred[m] - truth) for m in BUILDERS}
    rel = {m: weighted_norm(pred[m] - truth) / weighted_norm(truth) for m in BUILDERS}

    vmin, vmax = truth.min(), truth.max()
    emax = max(e.max() for e in error.values())
    panels = [
        (X[sample, :, 0].numpy(), X[sample, :, 0].min().item(), X[sample, :, 0].max().item(), "RdBu_r", f"{name}\nu(0)"),
        (truth, vmin, vmax, "RdBu_r", "true u(T)"),
        (pred["Ours"], vmin, vmax, "RdBu_r", f"Ours  (rel L2 {rel['Ours']:.3f})"),
        (pred["SFNO"], vmin, vmax, "RdBu_r", f"SFNO  (rel L2 {rel['SFNO']:.3f})"),
        (error["Ours"], 0, emax, "magma", "|Ours - true|"),
        (error["SFNO"], 0, emax, "magma", "|SFNO - true|"),
    ]
    for column, (values, lo, hi, cmap, title) in enumerate(panels):
        ax = fig.add_subplot(len(SYSTEMS), len(columns), row * len(columns) + column + 1, projection="3d")
        sphere_plot(ax, values, lo, hi, cmap, title)

plt.tight_layout()
plt.show()

## Where the errors live: error spectrum by degree

Because both models share the basis, we can ask *which* degrees $l$ each one gets
wrong. The plot shows the test-set error energy per degree,
$E(l)=\sum_m |\langle e, Y_{lm}\rangle|^2$. The point after $l=15$ is the error
energy outside the basis, which our model cannot reduce.

In [ ]:
fig, axes = plt.subplots(1, len(SYSTEMS), figsize=(4.2 * len(SYSTEMS), 3.4), sharey=False)
for ax, name in zip(axes, SYSTEMS):
    X, Yt = datasets[name]
    target = Yt[N_TRAIN:, :, 0].numpy()
    for model_name in BUILDERS:
        err = predict(models[name, model_name], model_name, X[N_TRAIN:])[:, :, 0].numpy() - target
        coefficients = err @ (w[:, None] * Y)                                    # [N_test, K]
        per_degree = np.array([(coefficients[:, degree == l] ** 2).sum() for l in range(LMAX + 1)])
        outside = (w * (err - coefficients @ Y.T) ** 2).sum()
        ax.semilogy(range(LMAX + 2), np.append(per_degree, outside), "o-", ms=3,
                    color=colors[model_name], label=model_name)
    ax.axvline(LMAX + 0.5, color="gray", lw=0.8, ls="--")
    ax.set_title(name)
    ax.set_xlabel("degree l   (last point: outside basis)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("test error energy")
axes[0].legend()
plt.tight_layout()
plt.show()

## Reading the results

- **Diffusion** is the cleanest test of the claim: the exact solution operator is
  $e^{tD\Delta}$, which is literally one of our heat kernels (and one SFNO
  per-degree weight). Both models should reach a very small error here. If ours
  does not, the problem is optimisation or initialisation, not the basis.
- **Reaction-diffusion** adds a weak nonlinearity on top of the same linear
  part.
- **Pendulum / Duffing** are purely *local* nonlinear maps. SFNO has pointwise
  skips $W x$ and a pointwise MLP decoder, so a local map is easy for it. Ours
  has to build it from smooth kernels plus the pairwise $\phi(f(x), f(y))$ term,
  and its output is band-limited (see the basis floor). A gap that appears only
  on the ODEs is therefore about the missing pointwise path, not the eigenbasis.
- The error spectra show whether a gap comes from low degrees (the model fits
  the large-scale structure badly) or high degrees (fine detail / the basis
  limit).